# LBP (tekstur) + Random Forest
        **vision › classification** · notebook `lbp_random_forest`

        **Kapan dipakai:** Klasifikasi tekstur (kain, permukaan, jaringan).

        **Jebakan umum:** Hanya tekstur lokal; lemah untuk objek.

**Data demo:** CIFAR-10 subset (4000 latih / 1000 uji, 10 kelas, 32x32). Bobot pretrained dibaca dari `pretrained/` (offline); GPU dipakai otomatis bila ada.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba. Format data: folder-per-kelas atau CSV `path,label`.

In [ ]:
DATA_ROOT  = ROOT / "data_latihan/vision/classification/cifar10_small"            # folder gambar
TRAIN_CSV  = DATA_ROOT / "train.csv"   # kolom: path (relatif ke DATA_ROOT), label
TEST_CSV   = DATA_ROOT / "test.csv"    # kolom: id, path (tanpa label)
TEST_LABELS = DATA_ROOT / "test_labels.csv"  # kunci jawaban test (hanya untuk latihan); None di lomba
SEED, VALID_SIZE = 42, 0.2
MODEL_KEY = "lbp_random_forest"
OUT = out_dir("vision", "classification", MODEL_KEY)
set_seed(SEED)

## 2. Baca gambar

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.vision import load_images, show_grid
tr = pd.read_csv(TRAIN_CSV); te = pd.read_csv(TEST_CSV)
le = LabelEncoder(); y = le.fit_transform(tr["label"]); CLASSES = list(le.classes_)
X = load_images(tr["path"], DATA_ROOT); Xte = load_images(te["path"], DATA_ROOT)
print(X.shape, Xte.shape, "| kelas:", CLASSES); show_grid(X[::400], [CLASSES[i] for i in y[::400]], n=10); plt.show()
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 3. Fitur & model
`featurize(X)` mengubah tiap gambar jadi vektor fitur manual; `model` mempelajarinya.

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from superdb.vision import hog_features, color_hist_features, lbp_features, pixel_features
from sklearn.ensemble import RandomForestClassifier

def featurize(X):
    return lbp_features(X)
model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

## 4. Ekstraksi fitur (CPU) + cross-validation + evaluasi

In [ ]:
import time; t0 = time.time()
Ftr, Fva = featurize(Xtr), featurize(Xva); print(f"fitur {Ftr.shape} dalam {time.time() - t0:.0f} dtk")
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = cross_val_score(model, Ftr, ytr, cv=StratifiedKFold(3, shuffle=True, random_state=SEED), scoring="f1_macro"); print(f"CV f1_macro = {cv.mean():.4f} ± {cv.std():.4f}")
model.fit(Ftr, ytr); pred = model.predict(Fva)
proba = model.predict_proba(Fva) if hasattr(model, "predict_proba") else None
metrics = classification_metrics(yva, pred, proba, classes=np.arange(len(CLASSES))); display(show_table(metrics))
plot_confusion(yva, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 5. Latih ulang pada semua data → submission → simpan

In [ ]:
from sklearn.base import clone
final = clone(model).fit(featurize(X), y); CLASSES_PRED = final.predict(featurize(Xte))

In [ ]:
pred_te = CLASSES_PRED
sub = pd.DataFrame({"id": te["id"], "label": [CLASSES[i] for i in pred_te]}); sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
if TEST_LABELS is not None and Path(TEST_LABELS).exists():
    truth = pd.read_csv(TEST_LABELS); acc = float((truth.set_index("id").loc[sub["id"], "label"].to_numpy() == sub["label"].to_numpy()).mean())
    print(f"akurasi pada test (kunci jawaban): {acc:.4f}"); metrics["test_accuracy"] = acc

In [ ]:
save_model({"model": final, "classes": CLASSES}, OUT / "model.joblib"); save_json({"cv_f1_macro": float(cv.mean()), "valid": metrics}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib"); assert (b["model"].predict(featurize(Xte[:30])) == final.predict(featurize(Xte[:30]))).all(); print("OK. hasil di", OUT)

## Langkah berikutnya
- Fitur manual cepat tetapi terbatas: bandingkan dengan `transfer_*` dan `finetune_*` di folder ini.
- Data lomba biasanya lebih besar/resolusi lebih tinggi: HOG pada ukuran asli, naikkan `pixels_per_cell`.